In [ ]:
import os, re, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score)

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)


In [ ]:
RESULTS = []

def log_result(experiment, domain_setting, model_family, model_name, metrics_dict):
    row = {
        "experiment": experiment,
        "domain_setting": domain_setting,
        "family": model_family,
        "model": model_name,
        "accuracy": metrics_dict.get("accuracy"),
        "precision": metrics_dict.get("precision"),
        "recall": metrics_dict.get("recall"),
        "f1": metrics_dict.get("f1"),
        "roc_auc": metrics_dict.get("roc_auc")
    }
    RESULTS.append(row)

def results_df():
    import pandas as pd
    return pd.DataFrame(RESULTS).sort_values(["experiment","family","model"]).reset_index(drop=True)




In [ ]:
LABEL_MAP = {"relevant": 1, "not_relevant": 0}

def load_tsv(filename, domain, split):
    try:
        df = pd.read_csv(filename, sep="\t", encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(filename, sep="\t", encoding="latin1") 

    df["y"] = df["label"].map(LABEL_MAP)
    if df["y"].isna().any():
        bad = df[df["y"].isna()]["label"].value_counts().head(10)
        raise ValueError(f"Unmapped labels in {filename}: {bad.to_dict()}")

    df["domain"] = domain
    
    return df[["tweet_id","text","label","y","domain"]]

nepal_train = load_tsv("2015_Nepal_Earthquake_train.tsv", "nepal_earthquake", "train")
nepal_dev = load_tsv("2015_Nepal_Earthquake_dev.tsv", "nepal_earthquake", "dev")
nepal_test  = load_tsv("2015_Nepal_Earthquake_test.tsv",  "nepal_earthquake", "test")
qld_train   = load_tsv("2013_Queensland_Floods_train.tsv", "qld_floods", "train")
qld_dev   = load_tsv("2013_Queensland_Floods_dev.tsv", "qld_floods", "dev")
qld_test    = load_tsv("2013_Queensland_Floods_test.tsv",  "qld_floods", "test")

#preparing dataset combining two different disasters domain
master_train = pd.concat([nepal_train,nepal_dev, qld_train, qld_dev], ignore_index=True)
master_test  = pd.concat([nepal_test,  qld_test],  ignore_index=True)

# print("Nepal train/test:", nepal_train.shape, nepal_test.shape)
# print("QLD train/test:", qld_train.shape, qld_test.shape)
print("Master train/test:", master_train.shape, master_test.shape)

master_train.head()


In [ ]:
master_test.head()

In [ ]:
# Load Chile Earthquake dataset
df = pd.read_csv('2014_Chile_Earthquake_en.csv', sep=",",
    encoding="latin-1",
    engine="python")

print(df.columns)

In [ ]:
df.columns = df.columns.str.strip()

# categories to be labeled as 0
category_zero = ['Personal updates, sympathy, support', 'Donations of money']

# chile_test dataframe
chile_test = pd.DataFrame()
chile_test['text'] = df['tweet_text']

chile_test['label'] = df['label'].apply(
    lambda x: 0 if x.strip() in category_zero else 1
)

chile_test['Domain'] = 'chile_earthquake'

chile_test.rename(columns={"label": "y"}, inplace=True)
chile_test.to_csv('chile_test.csv', index=False)

print(chile_test.head())
print(chile_test["y"].value_counts().rename({0: "Non-Disaster", 1: "Disaster"}))

### Data preprocessing, cleaning, EDA

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from wordcloud import WordCloud


# Text preprocessing 
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text)

    return text

def clean_tweet(text):
    text = str(text).lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", " ", text)

    # remove mentions
    text = re.sub(r"@\w+", " ", text)

    # remove hashtag symbol but keep word
    text = text.replace("#", "")

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Data stats after cleaning
def clean_dataset(df, name="dataset"):
    df = df.copy()
    original_size = len(df)

    missing = df["text"].isna().sum()
    df = df.dropna(subset=["text"])

    df["text"] = df["text"].apply(preprocess_text)
    df = df[df["text"].str.strip() != ""]

    before_dups = len(df)
    df = df.drop_duplicates(subset="text")
    duplicates_removed = before_dups - len(df)

    final_size = len(df)

    stats = {
        "Dataset": name,
        "Original": original_size,
        "Final": final_size,
        "Duplicates Removed": duplicates_removed,
        "Missing Removed": missing
    }

    return df.reset_index(drop=True), stats

# apply cleaning 

master_train_raw = master_train.copy()
master_test_raw  = master_test.copy()
nepal_train_raw  = nepal_train.copy()
qld_test_raw     = qld_test.copy()
nepal_test_raw = nepal_test.copy()
chile_test_raw = chile_test.copy()


master_train, stats_master_train = clean_dataset(master_train_raw, "Master Train")
master_test,  stats_master_test  = clean_dataset(master_test_raw,  "Master Test")

nepal_train,  stats_nepal_train  = clean_dataset(nepal_train_raw,  "Nepal Train")
qld_test,     stats_qld_test     = clean_dataset(qld_test_raw,     "QLD Test")
nepal_test,     stats_nepal_test     = clean_dataset(nepal_test_raw,     "Nepal Test")
chile_test,     stats_chile_test     = clean_dataset(chile_test_raw,     "Chile Test")


stats_df = pd.DataFrame([
    stats_master_train,
    stats_master_test,
    stats_nepal_train,
    stats_qld_test,
    stats_nepal_test,
    stats_chile_test
])

print("\n Dataset statistics ")
print(stats_df)

# Plots for EDA visualizations 

# Class distribution
def plot_class_distribution(df, title):
    df["y"].value_counts().plot(kind='bar')
    plt.title(f"Class Distribution: {title}")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()

plot_class_distribution(master_train, "Master Train")


# Tweet length distribution
def plot_length_distribution(df, title):
    lengths = df["text"].apply(lambda x: len(x.split()))
    plt.hist(lengths, bins=30)
    plt.title(f"Tweet Length Distribution: {title}")
    plt.xlabel("Number of Words")
    plt.ylabel("Frequency")
    plt.show()

plot_length_distribution(master_train, "Master Train")

#Top tokens by class
def plot_top_tokens_by_class(df, n=20):
    label_map = {0: "Non-Disaster", 1: "Disaster"}

    for cls in sorted(df["y"].unique()):
        subset = df[df["y"] == cls]

        tokens = " ".join(subset["text"]).split()

        # Remove noise tokens
        tokens = [t for t in tokens if t not in {"URL", "USER", "RT"}]

        # Remove stopwords
        tokens = [t for t in tokens if t.lower() not in ENGLISH_STOP_WORDS]

        # remove very short tokens
        tokens = [t for t in tokens if len(t) > 2]

        top_tokens = Counter(tokens).most_common(n)

        words, counts = zip(*top_tokens)

        plt.figure(figsize=(8,6))
        plt.barh(words, counts)
        plt.title(f"Top {n} Tokens - {label_map.get(cls, cls)}")
        plt.xlabel("Frequency")
        plt.ylabel("Tokens")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

plot_top_tokens_by_class(master_train)


# Plot for showing the stats of before and after cleaning
def plot_before_after(stats_df):
    x = range(len(stats_df))
    plt.bar(x, stats_df["Original"], color='steelblue', alpha=0.7, label="Original")
    plt.bar(x, stats_df["Final"], color='orange', alpha=0.7, label="Cleaned")

    plt.xticks(x, stats_df["Dataset"], rotation=45)
    plt.title("Dataset Size Before vs After Cleaning")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_before_after(stats_df)

# Word cloud
def plot_wordcloud_by_class(df):
    label_map = {0: "Non-Disaster", 1: "Disaster"}

    for cls in sorted(df["y"].unique()):
        subset = df[df["y"] == cls].copy() 

        text = " ".join(subset["text"])

        # fiter tokens for visualization
        tokens = text.split()

        # Remove noise 
        tokens = [t for t in tokens if t not in {"URL", "USER"}]

        # Remove stopwords
        tokens = [t for t in tokens if t.lower() not in ENGLISH_STOP_WORDS]

        text = " ".join(tokens)

        if not text.strip():
            continue

        wc = WordCloud(width=1200, height=600, background_color="white", collocations=False).generate(text)
        plt.figure(figsize=(12,5))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
       # plt.title(title)
        plt.title(f"Word Cloud - {label_map.get(cls, cls)}")
        plt.show()


plot_wordcloud_by_class(master_train)

#### Functions for result metrics

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score)

def plot_confusion(cm, title="Confusion matrix"):
    plt.figure(figsize=(4,4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks([0,1], ["0","1"]); plt.yticks([0,1], ["0","1"])
    for (i,j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.show()

def plot_roc_pr(y_true, y_score, title_prefix=""):
    out = {"roc_auc": None}
    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        plt.figure(figsize=(6,4))
        plt.plot(fpr, tpr)
        plt.plot([0,1],[0,1], linestyle="--")
        plt.title(f"{title_prefix} ROC (AUC={auc:.3f})")
        plt.xlabel("FPR"); plt.ylabel("TPR")
        #save_current_fig("ROC_" + title_prefix)
        plt.show()
        out["roc_auc"] = float(auc)
    except Exception as e:
        print("ROC unavailable:", e)

    return out


def summarize_metrics(y_true, y_pred, y_score=None, title="", verbose=True):
    acc = accuracy_score(y_true, y_pred)
    p,r,f,_ = precision_recall_fscore_support(
        y_true, y_pred, 
        average="binary", 
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    
    roc_metric = {"roc_auc": None}

    if verbose and y_score is not None:
        roc_metric = plot_roc_pr(y_true, y_score, title_prefix=title)

    if verbose:
        print(f"{title} Accuracy={acc:.4f} Precision={p:.4f} Recall={r:.4f} F1={f:.4f}")
        print(classification_report(y_true, y_pred, digits=4))
        plot_confusion(cm, title=f"{title}_ConfusionMatrix")

    return {
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f),
        "roc_auc": roc_metric.get("roc_auc"),
       
    }


### ML Models - SVM and Logistic Regression

In [ ]:
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV


def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


#pipelien for ML models
def run_ml_experiment(train_df, test_df, experiment="EXP", domain_setting=""):

    
    train_df = train_df.copy()
    test_df  = test_df.copy()

    # train-validation split
    train_split, val_split = train_test_split(
        train_df,
        test_size=0.1,
        stratify=train_df["y"],
        random_state=42
    )

    # model level text preprocessing
    train_split["text_model"] = train_split["text"].apply(clean_tweet)
    val_split["text_model"]   = val_split["text"].apply(clean_tweet)
    test_df["text_model"]     = test_df["text"].apply(clean_tweet)

    X_train, y_train = train_split["text_model"], train_split["y"].values
    X_val,   y_val   = val_split["text_model"],   val_split["y"].values
    X_test,  y_test  = test_df["text_model"],     test_df["y"].values

    # TF-IDF features
    tfidf_features = FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1,3),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
            stop_words="english",
            max_features=30000
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3,5),
            min_df=2,
            max_features=20000
        ))
    ])

    # SVM model
    base_svm = Pipeline([
        ("features", tfidf_features),
        ("svm", LinearSVC(C=1.0))
    ])

    svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=5)
    svm.fit(X_train, y_train)

    print("\n" + "="*80)
    print(f"{experiment}_{domain_setting}_svm_validation")

    y_val_pred = svm.predict(X_val)
    y_val_score = svm.predict_proba(X_val)[:,1]

    summarize_metrics(y_val, y_val_pred, y_val_score, title="Validation_SVM")

    print("\n" + "="*80)
    title = f"{experiment}_{domain_setting}_TFIDF_SVM"

    y_pred = svm.predict(X_test)
    y_score = svm.predict_proba(X_test)[:,1]

    m = summarize_metrics(y_test, y_pred, y_score, title=title)
    log_result(experiment, domain_setting, "ML", "TFIDF+SVM", m)

    
    # Logistic Regression model
   
    lr = Pipeline([
        ("features", tfidf_features),
        ("lr", LogisticRegression(
            max_iter=3000,
            C=1.2,
            n_jobs=-1
        ))
    ])

    lr.fit(X_train, y_train)

    
    print("\n" + "="*80)
    print(f"{experiment}_{domain_setting}_LR_VALIDATION")

    y_val_pred = lr.predict(X_val)
    y_val_score = lr.predict_proba(X_val)[:,1]

    #Show validation results
    summarize_metrics(y_val, y_val_pred, y_val_score, title="Validation_LR")

   
    print("\n" + "="*80)
    title = f"{experiment}_{domain_setting}_TFIDF_LogReg"

    y_pred = lr.predict(X_test)
    y_score = lr.predict_proba(X_test)[:,1]
    
    # after test metric and model results logging
    m = summarize_metrics(y_test, y_pred, y_score, title=title)
    log_result(experiment, domain_setting, "ML", "TFIDF+LogReg", m)

In [ ]:
run_ml_experiment(master_train, master_test, "EXP1", "Master_to_Master")
run_ml_experiment(master_train,  chile_test,  "EXP2", "Master_to_Chile")
run_ml_experiment(master_train, nepal_test, "EXP3", "Master_to_Nepal")
run_ml_experiment(master_train,  qld_test,  "EXP4", "Master_to_QLD")


### Classical DL models - TextCNN and BiLSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter

EPOCHS_DL = 10
BATCH_DL = 32

@torch.no_grad()
def eval_dl(model, loader, title="TEST", verbose=True):
    model.eval()
    all_y, all_p, all_s = [], [], []
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)

        probs = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
        pred = (probs >= 0.5).astype(int)

        all_y.append(yb.cpu().numpy())
        all_p.append(pred)
        all_s.append(probs)

    y_true=np.concatenate(all_y)
    y_pred=np.concatenate(all_p)
    y_score=np.concatenate(all_s)

    return summarize_metrics(
    y_true, y_pred, y_score,
    title=title,
    verbose=verbose
    )

def compute_val_loss(model, loader, criterion):
    model.eval()
    total = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        total += loss.item() * xb.size(0)

    return total / len(loader.dataset)

In [ ]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


def dl_tokenize(text: str):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()


# building vocabulary and converting tweets into numerical sequence

def build_vocab(texts, min_freq=2, max_size=50000):
    counter = Counter()
    for t in texts:
        counter.update(dl_tokenize(t))

    vocab = {"<PAD>":0, "<UNK>":1}
    for tok, freq in counter.most_common():
        if freq < min_freq:
            continue
        if tok in vocab:
            continue
        vocab[tok] = len(vocab)
        if len(vocab) >= max_size:
            break
    return vocab

MAX_LEN = 100  

vocab = build_vocab(master_train["text"].tolist())
VOCAB_SIZE = len(vocab)
print("Vocab size:", VOCAB_SIZE)

def encode(text, vocab, max_len=MAX_LEN):
    ids = [vocab.get(t, vocab["<UNK>"]) for t in dl_tokenize(text)][:max_len]
    if len(ids) < max_len:
        ids += [vocab["<PAD>"]] * (max_len - len(ids))
    return np.array(ids, dtype=np.int64)

#PyTorch dataset class to manage text and labels

class TextDataset(Dataset):
    def __init__(self, df, vocab):
        self.X = np.stack([encode(t, vocab) for t in df["text"].tolist()])
        self.y = df["y"].values.astype(np.int64)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i])

# loaders function with train and test data

def make_loaders(train_df, test_df, batch_size=BATCH_DL):
    train_ds = TextDataset(train_df, vocab)
    test_ds  = TextDataset(test_df, vocab)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False),
    )


# Add validation split in loaders

def make_loaders_with_val(train_df, test_df, batch_size=BATCH_DL):
    train_df_split, val_df = train_test_split(
        train_df,
        test_size=0.1,
        stratify=train_df["y"],
        random_state=42
    )

    train_ds = TextDataset(train_df_split, vocab)
    val_ds   = TextDataset(val_df, vocab)
    test_ds  = TextDataset(test_df, vocab)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False),
    )


# Load glove

glove_path = "glove.6B.100d.txt"

glove_dict = {}
with open(glove_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_dict[word] = vector

emb_dim = 100
glove_matrix = np.random.normal(scale=0.6, size=(VOCAB_SIZE, emb_dim)).astype(np.float32)
glove_matrix[vocab["<PAD>"]] = 0.0

found = 0
for tok, idx in vocab.items():
    if tok in ("<PAD>", "<UNK>"):
        continue
    if tok in glove_dict:
        glove_matrix[idx] = glove_dict[tok]
        found += 1

print(f"GloVe coverage: {found}/{VOCAB_SIZE} = {found/VOCAB_SIZE:.2%}")


# Attention layer 

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask=None):
        scores = self.attn(x).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return context


# Model - TextCNN with GloVe

class TextCNN_GloVe(nn.Module):
    def __init__(self, vocab_size, glove_matrix, kernel_sizes=(3,4,5,7), num_filters=128):
        super().__init__()

        emb_dim = glove_matrix.shape[1]

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(glove_matrix))
        self.embedding.weight.requires_grad = False

        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(emb_dim, num_filters, k),
                nn.ReLU(),
                nn.BatchNorm1d(num_filters)
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 2)

    def forward(self, x):
        emb = self.embedding(x).transpose(1,2)

        convs = [c(emb) for c in self.convs]
        pools = [torch.max(c, dim=2).values for c in convs]

        out = torch.cat(pools, dim=1)
        out = self.dropout(out)

        return self.fc(out)


# Model - BiLSTM with Trainable embedding

class BiLSTM_Attn(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.spatial_dropout = nn.Dropout2d(0.2)

        self.lstm1 = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden*2, hidden, batch_first=True, bidirectional=True)

        self.attn = Attention(hidden*2)

        self.fc = nn.Sequential(
            nn.Linear(hidden*2, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        emb = self.embedding(x)

        emb = emb.unsqueeze(1)
        emb = self.spatial_dropout(emb).squeeze(1)

        out, _ = self.lstm1(emb)
        out, _ = self.lstm2(out)

        mask = (x != 0)
        context = self.attn(out, mask)

        return self.fc(context)



# Model - BiLSTM with GloVe

class BiLSTM_GloVe_Attn(nn.Module):
    def __init__(self, vocab_size, glove_matrix, hidden=128):
        super().__init__()


        self.glove_used = True   
        self.embedding = nn.Embedding(vocab_size, glove_matrix.shape[1], padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(glove_matrix))
        self.embedding.weight.requires_grad = False  # freeze first

        self.spatial_dropout = nn.Dropout2d(0.2)

        self.lstm1 = nn.LSTM(glove_matrix.shape[1], hidden, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden*2, hidden, batch_first=True, bidirectional=True)

        self.attn = Attention(hidden*2)

        self.fc = nn.Sequential(
            nn.Linear(hidden*2, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        emb = self.embedding(x)

        emb = emb.unsqueeze(1)
        emb = self.spatial_dropout(emb).squeeze(1)

        out, _ = self.lstm1(emb)
        out, _ = self.lstm2(out)

        mask = (x != 0)
        context = self.attn(out, mask)

        return self.fc(context)


# Model training pipeline

import copy

def fit_dl(model, train_loader, val_loader, test_loader, title, epochs=EPOCHS_DL, lr=1e-3):
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=2, factor=0.5
    )

   
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0
    patience = 2
    counter = 0
    best_model = None

    for ep in range(1, epochs+1):
        model.train()
        total = 0

        for xb, yb in tqdm(train_loader, leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total += loss.item() * xb.size(0)

        tr_loss = total / len(train_loader.dataset)

        # validation loss
        val_loss = compute_val_loss(model, val_loader, criterion)
        scheduler.step(val_loss)

        print(f"Epoch {ep}/{epochs} train_loss={tr_loss:.4f} val_loss={val_loss:.4f}")

        # unfreezing GloVe after few epochs
        if ep == 5 and hasattr(model, "embedding") and hasattr(model, "glove_used"):
            print("Unfreezing GloVe embeddings...")
            model.embedding.weight.requires_grad = True

        # validation metrics
        val_metrics = eval_dl(model, val_loader, title="VAL", verbose=False)
        val_f1 = val_metrics["f1"]

        print(f"Validation F1: {val_f1:.4f}")

        # early stopping
        if val_f1 > best_f1:
            best_f1 = val_f1
            counter = 0
            best_model = copy.deepcopy(model)
        else:
            counter += 1
            if counter >= patience:
                print("triggered early stopping ")
                break

    print(f"Best Validation F1: {best_f1:.4f}")

    model = best_model if best_model is not None else model

    metrics = eval_dl(model, test_loader, title=title, verbose=True)
    return model, metrics


def run_classical_dl(train_df, test_df, experiment, domain_setting):
    train_loader, val_loader, test_loader = make_loaders_with_val(train_df, test_df)

    
    # CNN with GloVe
    
    title = f"{experiment}_{domain_setting}_TextCNN_GloVe"
    print("\n" + "="*80); print(title)

    model = TextCNN_GloVe(VOCAB_SIZE, glove_matrix)

    _, m = fit_dl(model, train_loader, val_loader, test_loader, title=title)

    log_result(experiment, domain_setting, "ClassicalDL",
               "TextCNN+GloVe", m)
   
    # BiLSTM 
    
    title = f"{experiment}_{domain_setting}_BiLSTM_Attn"
    print("\n" + "="*80); print(title)

    model = BiLSTM_Attn(VOCAB_SIZE)

    _, m = fit_dl(model, train_loader, val_loader, test_loader, title=title)

    log_result(experiment, domain_setting, "ClassicalDL",
           "BiLSTM+Attention", m)

   
    # BiLSTM with GloVe
    
    title = f"{experiment}_{domain_setting}_BiLSTM_GloVe_Attn"
    print("\n" + "="*80); print(title)

    model = BiLSTM_GloVe_Attn(VOCAB_SIZE, glove_matrix)

    _, m = fit_dl(model, train_loader, val_loader, test_loader, title=title)

    log_result(experiment, domain_setting, "ClassicalDL",
               "BiLSTM+GloVe+Attention", m)



In [ ]:
run_classical_dl(master_train, master_test, "EXP1", "Master_to_Master")
run_classical_dl(master_train,  chile_test,  "EXP2", "Master_to_Chile")
run_classical_dl(master_train, nepal_test, "EXP3", "Master_to_Nepal")
run_classical_dl(master_train,  qld_test,  "EXP4", "Master_to_QLD")


### Transformers and Hybrid of Transformer-RNN

In [ ]:
EPOCHS_FINETUNE = 3
EPOCHS_HYBRID = 2
BATCH_HF = 8
MAX_LEN_HF = 128

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup


def bert_clean(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\s+", " ", text)

    return text

# Hugging Face dataset class to preprocess and tokenize tweets through the Hugging Face tokenizer
class HFDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN_HF):
        self.texts = df["text"].apply(bert_clean).tolist()
        self.y = df["y"].values.astype(int)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.y)   #returns total samples

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.y[idx], dtype=torch.long)
        return item



def hf_loaders(train_df, test_df, model_name, batch_size=BATCH_HF):
    tok = AutoTokenizer.from_pretrained(model_name)

    train_ds = HFDataset(train_df, tok)
    test_ds  = HFDataset(test_df, tok)

    return (
        tok,
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False),
    )
    

@torch.no_grad()
def eval_hf_logits(get_logits, loader, title="TEST"):
    
    true_label = []
    pred_label = []
    pred_score = []
    
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        am  = batch["attention_mask"].to(DEVICE)
        y   = batch["labels"].to(DEVICE)

        logits = get_logits(ids, am)

        probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        pred  = (probs >= 0.5).astype(int)

        true_label.append(y.cpu().numpy())
        pred_label.append(pred)
        pred_score.append(probs)

    return summarize_metrics(
        np.concatenate(true_label),
        np.concatenate(pred_label),
        np.concatenate(pred_score),
        title=title,
        verbose=True
    )

# BERT training 

def fit_finetune(model_name, train_df, test_df, title, epochs=EPOCHS_FINETUNE, lr=2e-5):

    tok, train_loader, test_loader = hf_loaders(train_df, test_df, model_name)

    #Load pre-trained transformer
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps
    )

    criterion = nn.CrossEntropyLoss()

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            ids = batch["input_ids"].to(DEVICE)
            am  = batch["attention_mask"].to(DEVICE)
            y   = batch["labels"].to(DEVICE)

            logits = model(input_ids=ids, attention_mask=am).logits
            loss = criterion(logits, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            total += loss.item() * ids.size(0)

        print(f"Epoch {ep}/{epochs} train_loss={total / len(train_loader.dataset):.4f}")

    model.eval()
    metrics = eval_hf_logits(
        lambda ids, am: model(input_ids=ids, attention_mask=am).logits,
        test_loader,
        title=title
    )

    return tok, model, metrics


class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, x, mask):
        scores = self.proj(x).squeeze(-1)
        scores = scores.masked_fill(~mask.bool(), -1e9)
        attn = torch.softmax(scores, dim=1)
        return torch.bmm(attn.unsqueeze(1), x).squeeze(1)

# Hybrid BERT-BiLSM model

class BERT_BiLSTM_Attn(nn.Module):
    def __init__(self, base_model="bert-base-uncased", hidden=128, dropout=0.3):
        super().__init__()

        self.base = AutoModel.from_pretrained(base_model)
        bert_dim = self.base.config.hidden_size

        self.lstm = nn.LSTM(
            bert_dim, 
            hidden, 
            batch_first=True, 
            bidirectional=True)
        
        self.attn = AttentionPool(hidden * 2)

        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, 2)

    def forward(self, input_ids, attention_mask):
        x = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.lstm(x)
        lstm_out = self.drop(lstm_out)
        pooled = self.attn(lstm_out, attention_mask)
        return self.fc(self.drop(pooled))



# Hybrid BERT-BiLSTM-Attn training

def fit_hybrid(train_df, test_df, title, epochs=EPOCHS_HYBRID):

    tok, train_loader, test_loader = hf_loaders(train_df, test_df, "bert-base-uncased")

    model = BERT_BiLSTM_Attn().to(DEVICE)

    optimizer = torch.optim.AdamW([
        {"params": model.base.parameters(), "lr": 2e-5},
        {"params": model.lstm.parameters(), "lr": 5e-4},
        {"params": model.attn.parameters(), "lr": 5e-4},
        {"params": model.fc.parameters(), "lr": 5e-4},
    ])

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps
    )

    criterion = nn.CrossEntropyLoss()

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            ids = batch["input_ids"].to(DEVICE)
            am  = batch["attention_mask"].to(DEVICE)
            y   = batch["labels"].to(DEVICE)

            loss = criterion(model(ids, am), y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            total += loss.item() * ids.size(0)

        print(f"Epoch {ep}/{epochs} train_loss={total / len(train_loader.dataset):.4f}")

    model.eval()
    metrics = eval_hf_logits(lambda ids, am: model(ids, am), test_loader, title=title)

    return tok, model, metrics


# function to run the models 

def run_transformers(train_df, test_df, experiment, domain_setting):


    # BERT
    _, _, m = fit_finetune(
        "bert-base-uncased",
        train_df,

        test_df,
        f"{experiment}_{domain_setting}_BERT"
    )

    log_result(experiment, domain_setting, "Transformer", "BERT", m)

    # HYBRID
    _, _, m = fit_hybrid(
        train_df,
        test_df,
        f"{experiment}_{domain_setting}_Hybrid"
    )

    log_result(experiment, domain_setting, "Hybrid", "BERT+BiLSTM+Attn", m)

In [ ]:
run_transformers(master_train, master_test, "EXP1", "Master_to_Master")


In [ ]:

run_transformers(master_train,  chile_test,  "EXP2", "Master_to_Chile")



In [ ]:
run_transformers(master_train, nepal_test, "EXP3", "Master_to_Nepal")



In [ ]:
run_transformers(master_train,  qld_test,  "EXP4", "Master_to_QLD")


### Summary Results - Plots

In [ ]:
COLOR_FAMILY = {
    "ClassicalDL": "#1f77b4",  # blue
    "ML": "#ff7f0e",           # orange
    "Transformer": "#2ca02c",  # green 
    "Hybrid": "#d62728"        # red 
}


EXP_LABELS = {
    "EXP1": "In-Domain (Master)",
    "EXP2": "Domain Shift (Chile)",
    "EXP3": "Nepal",
    "EXP4": "Queensland"
}



MODEL_MAP = {
    "TFIDF+SVM": "SVM",
    "TFIDF+LogReg": "LogReg",
    "TextCNN+GloVe": "TextCNN-GloVe",
    "BiLSTM+Attention": "BiLSTM",
    "BiLSTM+GloVe+Attention": "BiLSTM-GloVe",
    "BERT+BiLSTM+Attn" : "BERT-BiLSTM"
}

COLORS = {
    "EXP1": "#1f77b4",
    "EXP3": "#2ca02c",
    "EXP4": "#d62728",
    "EXP2": "#ff7f0e"
}


In [ ]:
def get_results_df():
    return results_df().copy()

In [ ]:
#each modeling pipeline-wise plot

def plot_each_pipeline(family="ML", metric="f1"):
    df = results_df()
    df = df[df["family"] == family].copy()


    df["model"] = df["model"].replace(MODEL_MAP)

    EXP_ORDER = ["EXP1", "EXP3", "EXP4", "EXP2"]

    pivot = df.pivot(index="model", columns="experiment", values=metric)
    pivot = pivot[EXP_ORDER]

    ax = pivot.plot(
        kind="bar",
        figsize=(7,5),
        color=[COLORS[e] for e in EXP_ORDER]
    )

    plt.title(f"{family} Models ({metric.upper()})")
    plt.ylabel(metric.upper())
    plt.xlabel("Model")

    plt.xticks(rotation=0)

    # to show legends below the plot horizontally
    handles, labels = ax.get_legend_handles_labels()
    labels = [EXP_LABELS.get(l, l) for l in labels]

    plt.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=len(labels),
        frameon=False
    )

    plt.subplots_adjust(bottom=0.25)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_best_model_per_pipeline(metric="f1"):
    import matplotlib.pyplot as plt

    df = results_df().copy()

   
    MODEL_MAP = {
        "TFIDF+LogReg": "LogReg",
        "TFIDF+SVM": "SVM",
        "TextCNN+GloVe": "TextCNN",
        "BiLSTM+Attention": "BiLSTM",
        "BiLSTM+GloVe+Attention": "BiLSTM-GloVe",
        "BERT": "BERT",
        "BERT+BiLSTM+Attn": "BERT-BiLSTM"
    }

    # best model per experiment and model pipeline
    best = (
        df.sort_values(metric, ascending=False)
          .groupby(["experiment", "family"])
          .first()
          .reset_index()
    )

    best["model"] = best["model"].map(MODEL_MAP).fillna(best["model"])
    best["experiment"] = best["experiment"].map(EXP_LABELS)

    
    pivot = best.pivot(index="experiment", columns="family", values=metric)

   
    fig, ax = plt.subplots(figsize=(14,6))

    pivot.plot(
        kind="barh",
        ax=ax,
        color=[COLOR_FAMILY[f] for f in pivot.columns],
        edgecolor="#b0b0b0",
        linewidth=0.8
    )

    
    ax.set_title(f"Best Model per Pipeline ({metric.upper()})", fontsize=14)
    ax.set_xlabel(metric.upper(), fontsize=12)
    ax.set_ylabel("")

    ax.set_xlim(0, 1.08)

   
    ax.grid(axis="x", linestyle="--", alpha=0.3, color="#b0b0b0")
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_edgecolor("#b0b0b0")
        spine.set_linewidth(1)

   
    offsets = {
        fam: i*0.2 - (len(pivot.columns)-1)*0.1
        for i, fam in enumerate(pivot.columns)
    }

    
    for _, row in best.iterrows():
        exp = row["experiment"]
        fam = row["family"]
        model = row["model"]
        val = row[metric]

        y_pos = list(pivot.index).index(exp)

        ax.text(
            val + 0.01,
            y_pos + offsets[fam],
            model,
            va='center',
            ha='left',
            fontsize=8,
            color="black",
            #weight="semibold"
        )

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=len(pivot.columns),
        frameon=False,
        fontsize=10
    )

   
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.22)

    plt.show()

In [ ]:
# All models comparison on F1 score

def plot_model_comparison(metric="f1"):
    df = results_df().copy()

    
    df["model"] = df["model"].replace(MODEL_MAP)

   
    pivot = df.pivot_table(
        index="model",
        columns="experiment",
        values=metric
    )

    colors = [COLORS.get(col, "gray") for col in pivot.columns]

    ax = pivot.plot(
        kind="barh",
        figsize=(10,7),
        color=colors,
        width=0.8
    )

    plt.title(f"{metric.upper()} Comparison Across Models")
    plt.xlabel(metric.upper())
    plt.ylabel("Model")

    # to add value labels
    for i, model in enumerate(pivot.index):
        for j, exp in enumerate(pivot.columns):
            val = pivot.loc[model, exp]

            # vertical offset for grouped bars
            y_pos = i + (j - (len(pivot.columns)-1)/2) * 0.2

            ax.text(
                val + 0.01,                
                y_pos,
                f"{val:.2f}",              
                va='center',
                ha='left',
                fontsize=8,
                color="black"
            )

   
    ax.set_xlim(0, 1.08)

    # legend formatting
    handles, labels = ax.get_legend_handles_labels()
    labels = [EXP_LABELS.get(l, l) for l in labels]

    plt.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=len(labels),
        frameon=False
    )

    plt.subplots_adjust(bottom=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
df_all = get_results_df()

# plots for individual pipeline
plot_each_pipeline("ML")
plot_each_pipeline("ClassicalDL")
plot_each_pipeline("Transformer")
plot_each_pipeline("Hybrid")

# plot best model in each pipeline and experiment
plot_best_model_per_pipeline()

# Plot for all models
plot_model_comparison()

### Results - Tabular Charts

In [ ]:
#Tabular chart for each pipeline

METRIC_ORDER = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def results_table(family="ML"):
    df = results_df().copy()
    df = df[df["family"] == family]

    df["model"] = df["model"].replace(MODEL_MAP)

   
    df = df[[
        "model", "experiment",
        "accuracy", "precision", "recall", "f1", "roc_auc"
    ]]

    table = df.pivot_table(
        index="model",
        columns="experiment",
        values=METRIC_ORDER
    )

    
    table = table.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

    
    table = table.reindex(METRIC_ORDER, axis=1, level=1)

   
    EXP_LABELS = {
        "EXP1": "In-Domain(Master Test)",
        "EXP2": "Domain Shift(Chile Earthquake)",
        "EXP3": "In-Domain(Nepal Earthquake)",
        "EXP4": "In-Domain(Queensland Flood)"
    }

    METRIC_LABELS = {
        "accuracy": "ACC",
        "precision": "PRE",
        "recall": "REC",
        "f1": "F1",
        "roc_auc": "AUC"
    }

    table.columns = table.columns.set_levels(
        [EXP_LABELS.get(col, col) for col in table.columns.levels[0]],
        level=0
    )

    table = table.rename(columns=METRIC_LABELS, level=1)

  
    return (
        table.round(3)
        .style
        .format("{:.3f}")
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]}
        ])
    )

In [ ]:
# ML pipeline tabular chart

results_table("ML")

In [ ]:
# Classical DL pipeline tabular chart

results_table("ClassicalDL")

In [ ]:
results_table("Transformer")

In [ ]:
results_table("Hybrid")

In [ ]:
METRIC_ORDER = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def results_table_all_models():
    df = results_df().copy()

    df["model"] = df["model"].replace(MODEL_MAP)

    df = df[[
        "model", "experiment",
        "accuracy", "precision", "recall", "f1", "roc_auc"
    ]]

    # pivot (same structure as before)
    table = df.pivot_table(
        index="model",
        columns="experiment",
        values=METRIC_ORDER
    )

    # EXP first
    table = table.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

    # enforce metric order
    table = table.reindex(METRIC_ORDER, axis=1, level=1)

    # rename experiments
    EXP_LABELS = {
        "EXP1": "In-Domain(Master Test)",
        "EXP2": "Domain Shift(Chile Earthquake)",
        "EXP3": "In-Domain(Nepal Earthquake)",
        "EXP4": "In-Domain(Queensland Flood)"
    }

    table.columns = table.columns.set_levels(
        [EXP_LABELS.get(col, col) for col in table.columns.levels[0]],
        level=0
    )

    # rename metrics
    METRIC_LABELS = {
        "accuracy": "ACC",
        "precision": "PRE",
        "recall": "REC",
        "f1": "F1",
        "roc_auc": "AUC"
    }

    table = table.rename(columns=METRIC_LABELS, level=1)

    table = table.sort_index()

    return (
        table.round(3)
        .style
        .format("{:.3f}")
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "center")]}
        ])
    )

In [ ]:
results_table_all_models()